 # Airport with most delay on average for each carrier

In [1]:
import org.apache.spark

Intitializing Scala interpreter ...

Spark Web UI available at http://LAPTOP-NJF4DDR7.station:4040
SparkContext available as 'sc' (version = 3.5.1, master = local[*], app id = local-1739539033915)
SparkSession available as 'spark'


import org.apache.spark


In [ ]:
// DO NOT EXECUTE - this is needed just to avoid showing errors in the following cells
val sc = spark.SparkContext.getOrCreate()

## Parsing the data

Firstly, we need some functions that can parse our CSV files.

In [2]:
def getInt(str:String) : Int = {
    if (str.forall(Character.isDigit))
        str.toInt
    else
        -1
}

def parse_flight(line: String) = {
    val parts = line.split(",")
    val year = getInt(parts(0))
    val month = getInt(parts(1))
    val day = getInt(parts(2))
    val dep_time = parts(4)
    val dep_delay = getInt(parts(15))
    val arr_time = parts(6)
    val arr_delay = getInt(parts(14))
    val carrier = parts(8)
    val flightnum = parts(9)
    val tailnum = parts(10)
    val origin = parts(16)
    val dest = parts(17)
    (year, month, day, dep_time, dep_delay, arr_time, arr_delay, carrier, tailnum, flightnum, origin, dest)
}

def parse_carrier(line: String) = {
    val parts = line.split(",").map(_.trim.replaceAll("^\"|\"$", ""))
    val code = parts(0).toUpperCase
    val description = parts(1)
    (code, description)
}

def parse_airport(line: String) = {
    val parts = line.split(",").map(_.trim.replaceAll("^\"|\"$", ""))
    val iata = parts(0).toUpperCase
    val airport = parts(1)
    val city = parts(2)
    val state = parts(3)
    val country = parts(4)
    val lat = parts(5)
    val long = parts(6)
    (iata, airport, city, state, country, lat, long)
}


getInt: (str: String)Int
parse_flight: (line: String)(Int, Int, Int, String, Int, String, Int, String, String, String, String, String)
parse_carrier: (line: String)(String, String)
parse_airport: (line: String)(String, String, String, String, String, String, String)


## Loading the data

Then we can load our CSV files as RDD's. Then we map with the index to get rid of the CSV file headers and parse flight and aircraft records.

In [ ]:
val flightsPath = "./../../../../datasets/project/flights.csv"
val airportsPath = "./../../../../datasets/project/airports.csv"
val carriersPath = "./../../../../datasets/project/carriers.csv"

val rddFlights = sc.textFile(flightsPath).mapPartitionsWithIndex { (idx, iter) => if (idx == 0) iter.drop(1) else iter }.map(parse_flight)
val rddAirports = sc.textFile(airportsPath).mapPartitionsWithIndex { (idx, iter) => if (idx == 0) iter.drop(1) else iter }.map(parse_airport)
val rddCarriers = sc.textFile(carriersPath).mapPartitionsWithIndex { (idx, iter) => if (idx == 0) iter.drop(1) else iter }.map(parse_carrier)

flightsPath: String = ./../../../../datasets/project/2008.csv/2008.csv
airportsPath: String = ./../../../../datasets/project/airports.csv
carriersPath: String = ./../../../../datasets/project/carriers.csv
rddFlights: org.apache.spark.rdd.RDD[(Int, Int, Int, String, Int, String, Int, String, String, String, String, String)] = MapPartitionsRDD[3] at map at <console>:32
rddAirports: org.apache.spark.rdd.RDD[(String, String, String, String, String, String, String)] = MapPartitionsRDD[7] at map at <console>:33
rddCarriers: org.apache.spark.rdd.RDD[(String, String)] = MapPartitionsRDD[11] at map at <console>:34


### Step 1: Extract relevant data (carrier, origin, arr_delay)

In [4]:
val carrierAirportDelays = rddFlights
  .map { case (_, _, _, _, _, _, arr_delay, carrier, _, _, origin, _) => 
    ((carrier, origin), (arr_delay, 1)) 
  }

carrierAirportDelays: org.apache.spark.rdd.RDD[((String, String), (Int, Int))] = MapPartitionsRDD[12] at map at <console>:26


### Step 2: Calculate average delay per (carrier, origin) pair

In [5]:
val avgDelayPerAirportCarrier = carrierAirportDelays
  .reduceByKey { case ((sumDelay1, count1), (sumDelay2, count2)) =>
    (sumDelay1 + sumDelay2, count1 + count2)
  }
  .mapValues { case (totalDelay, count) => totalDelay.toDouble / count }

avgDelayPerAirportCarrier: org.apache.spark.rdd.RDD[((String, String), Double)] = MapPartitionsRDD[14] at mapValues at <console>:29


### Step 3: Find the airport with the highest delay per carrier

In [6]:
val worstAirportForEachCarrier = avgDelayPerAirportCarrier
  .map { case ((carrier, airport), avgDelay) => (carrier, (airport, avgDelay)) }
  .reduceByKey { case ((airport1, delay1), (airport2, delay2)) =>
    if (delay1 > delay2) (airport1, delay1) else (airport2, delay2)
  }


worstAirportForEachCarrier: org.apache.spark.rdd.RDD[(String, (String, Double))] = ShuffledRDD[16] at reduceByKey at <console>:27


### Step 4: Join with airport and carrier names

In [7]:
val carrierPairs = rddCarriers.map(x => (x._1.trim.toUpperCase, x._2.trim))
val airportPairs = rddAirports.map(x => (x._1.trim.toUpperCase, x._2.trim))

val withCarrierNames = worstAirportForEachCarrier
  .map { case (carrier, (airport, avgDelay)) => (carrier.trim.toUpperCase, (airport.trim.toUpperCase, avgDelay)) }
  .join(carrierPairs)

val withFullNames = withCarrierNames
  .map { case (carrierCode, ((airportCode, avgDelay), carrierName)) => (airportCode, (carrierCode, carrierName, avgDelay)) }
  .join(airportPairs)

val results = withFullNames.map {
  case (airportCode, ((carrierCode, carrierName, avgDelay), airportName)) =>
    (s"$carrierCode ($carrierName)", s"$airportCode ($airportName)", avgDelay)
}

carrierPairs: org.apache.spark.rdd.RDD[(String, String)] = MapPartitionsRDD[17] at map at <console>:27
airportPairs: org.apache.spark.rdd.RDD[(String, String)] = MapPartitionsRDD[18] at map at <console>:28
withCarrierNames: org.apache.spark.rdd.RDD[(String, ((String, Double), String))] = MapPartitionsRDD[22] at join at <console>:32
withFullNames: org.apache.spark.rdd.RDD[(String, ((String, String, Double), String))] = MapPartitionsRDD[26] at join at <console>:36
results: org.apache.spark.rdd.RDD[(String, String, Double)] = MapPartitionsRDD[27] at map at <console>:38


### Step 5: Save results to file

In [8]:
results.coalesce(1).saveAsTextFile("output")